In [ ]:
!pip install transformers==4.46.0 accelerate==1.1.1

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 3.0 MB/s eta 0:00:00
Reason for being yanked: This version unfortunately does not work with 3.8 but we did not drop the support yet
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 59.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 333.2/333.2 kB 18.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 23.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 30.5 MB/s eta 0:00:00
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.28.0
    Uninstalling huggingface_hub-1.28.0:
      Successfully uninstalled huggingface_hub-1.28.0
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.22.2
    Uninstalling tokenizers-0.22.2:
      Successfully uninstalled tokenizers-0.22.2
  Attempting uninstall: transformers
    Found existing installation: transformers 5.15.1
    Uninstalling transformers-5.15.1:
 

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL = "Qwen/Qwen2.5-1.5B-Instruct"

tok = AutoTokenizer.from_pretrained(MODEL)
tok.pad_token = tok.eos_token
tok.padding_side = "left"

model = AutoModelForCausalLM.from_pretrained(
    MODEL,
    torch_dtype=torch.float16,
    device_map="cuda",
)

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

In [17]:
import time
import threading
from transformers import TextIteratorStreamer

def prompt_of_len(n_tokens: int) -> str:
    base = "Explain the following in detail.\n"

    filler_unit = (
        "A data center serves many inference "
        "requests at once. "
    )

    text = base

    while len(tok(text)["input_ids"]) < n_tokens:
        text += filler_unit

    ids = tok(text)["input_ids"][:n_tokens]

    assert len(ids) == n_tokens, (
        f"only produced {len(ids)} tokens, "
        f"wanted {n_tokens}"
    )

    return tok.decode(ids)

def measure_stream(prompt: str, new_tokens: int = 128):
    enc = tok(prompt, return_tensors="pt").to("cuda")

    streamer = TextIteratorStreamer(
        tok,
        skip_prompt=True,
        skip_special_tokens=True,
    )

    kwargs = dict(
        **enc,
        max_new_tokens=new_tokens,
        do_sample=False,
        streamer=streamer,
    )

    th = threading.Thread(
        target=model.generate,
        kwargs=kwargs,
    )

    t0 = time.time()
    th.start()

    stamps = []
    for _ in streamer:
        stamps.append(time.time())

    th.join()

    ttft = stamps[0] - t0

    if len(stamps) > 1:
        gaps = [b - a for a, b in zip(stamps, stamps[1:])]
        tpot = sum(gaps) / len(gaps)
    else:
        tpot = 0.0

    total = stamps[-1] - t0

    return {
        "ttft_s": round(ttft, 4),
        "tpot_s": round(tpot, 4),
        "total_s": round(total, 4),
        "n_tokens": len(stamps),
    }

In [18]:
for n in [128, 512, 2048, 4096]:
    p = prompt_of_len(n)
    actual = len(tok(p)["input_ids"])

    assert actual == n, (
        f"prompt_of_len({n}) "
        f"produced {actual} tokens"
    )

print("GREEN CHECK: PASS")

GREEN CHECK: PASS
